# Sociolinguistics : observing linguistic variation in social context

In this Assignment we want to focus on a task you might be interested in if you are studying sociolinguistics or social variation in language use. We focus on language variation in relation to specific identity marker: age. We already manually annotated some data and found some potential lexical choices or grammatical constructions that might be more frequent in younger versus older speakers. 

To have some data to study the differences in language between people of younger versus older age groups we need texts with speaker information. As an approximation we use two different Reddit communities (sub-reddits) that are focused on different age groups:
- r/GenZ (a community for members of Generation Z)
- r/AskOldPeople (a community for older adults to share their experiences and advice)

Of course we cannot be sure that all members of these communities belong to the respective age groups, but we can assume that there will be a lot of overlap.

The goal of this assignment is to find out differences in language use between these two communities.

As practical methods we want to learn:
- how to add additional information to an existing corpus object
- how to extract linguistic features that are useful to study language variation
- how to compare the two communities based on these features

## Part 1 of the Assignment:

Before we can do the analysis we will conduct a few pre-processing steps. We will focus on this part first and then do the analysis.
(1) Tokenization of the corpus
(2) Esxtract different features
(3) Create new corpus objects that contain utterances and the new type of information.

## Loading the Reddit Corpus in Convokit
The Convokit library provides a ready-to-use Reddit corpus that we can use for our analysis. Instead of loading the full corpus, we can load specific sub-reddits. For our analysis we want to load the two sub-reddits mentioned above: r/GenZ and r/AskOldPeople.

*Task*: Load the two different datasets and compare the size of the two corpora (number of speakers and number of utterances).
Add tokenization as a pre-processing, since we want to use number of tokens as information to filter out reddit posts that are very short or very long. Use the TextParser preprocessor for this and add this transformation to both corpora. You might need to manually download nltk resources if you get an error (nltk.download('punkt_tab')). The tokenization might take a while depending on your machine (it took ~7min on my laptop for both corpora).



In [1]:
from convokit import Corpus, download
corpus_old= Corpus(filename=download("subreddit-AskOldPeople"))
corpus_young = Corpus(filename=download("subreddit-GenZ"))



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/falkne/PycharmProjects/nlpforSocialInteractions/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/falkne/PycharmProjects/nlpforSocialInteractions/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/falkne/PycharmProjects/nlpforSocialInteractions/.venv/lib

Dataset already exists at /Users/falkne/.convokit/downloads/subreddit-AskOldPeople
Dataset already exists at /Users/falkne/.convokit/downloads/subreddit-GenZ


## Filter out very short and very long posts
After pre-processing the tokens should be stored in the meta.parsed field.
 
*Task*: Add another meta information field 'num_tokens' that contains the number of tokens in each post. You can iterate over the utterances in each corpus and count the number of tokens based on the meta.parsed field. Add a new metadata field to each utterance object ('meta.num_tokens') that contains the number of tokens. Then filter out utterances that have less than 11 tokens or more than 349 tokens and store the filtered utterances in a dataframe. How much data is left after filtering?

## Create a new corpus object with filtered utterances.

We now want to create a new corpus object in which we want to store all our relevant information for the analysis. The new corpus objects should only contain the filtered utterances. You can create a new corpus object in Convokit by providing utterance objects to the Corpus constructor:
Corpus(utterances=list_of_utterance_objects).

*Task*: Create two new corpus objects (one for each age group) that only contain the filtered utterances.

## Extract linguistic features
Now we want to extract some linguistic features that we can use to compare the two age groups. We will focus on two types of features:
- **Lexical features:** We will extract the frequency of specific expressions that might be distinctive for the Gen Z generation (e.g., slang words or specific phrases).
- **Syntactic features:** We will extract the frequency of specific POS tags (e.g. n_adj is the amount of adjectives in an utterance) and for each POS tag how many unique words the utterance contains (e.g. maybe a person uses a lot of adjectives but always the same one). The unique POS tags thus measure lexical variation within a certain syntactic category. The unique number of adjectives would be denoted by n_uadj. 
- **linguistic complexity:** we use corrected type token ratio (CTTR) as a measure of lexical diversity in an utterance. CTTR is calculated as the number of unique tokens divided by the square root of two times the total number of tokens. The higher the CTTR, the more diverse the vocabulary used in the utterance. We look at the average number of words per sentence to get an idea of whether a speaker uses longer or shorter sentences on average. And we look at the average age of aquisition for the words used in the utterance (based on a predefined lexicon that contains age of aquisition ratings for words). Higher values indicate that the speaker uses words that are typically learned later in life, which can be an indicator of linguistic sophistication.

## Lexical features
I provided a word list of Gen Z unigrams / n-grams that you can use to extract lexical features. The file is called 'genz_wordlist.txt' and contains one word or n-gram per line. Feel free to add more words that you think are relevant for the analysis. 
*Task*: Load the word list and extract the total number of Gen-Z words used in each utterance. Store this information in a new meta field 'meta.num_genz_words' for each utterance.

## Syntactic features and linguistic complexity

To extract these features we use an existing library. The library is called *LFTK* : https://lftk.readthedocs.io/en/latest/
It provides a list of features that can be extracted from text, including POS tag frequencies and measures of lexical diversity.
 
As a first step we will extract the feature names of the features we want to look at. One can use the "search_features" function to search for features based on domain and family. a in a_ stands for average, and n_ stands for number of occurrences. ps stands for per sentence.

In [3]:
import lftk
pos_features = lftk.search_features(domain = 'syntax', family = "partofspeech", language="general", return_format = "list_dict")
pos_features = [f['key'] for f in pos_features]
additional_features = ["a_word_ps", "a_bry_ps", "corr_ttr"]
features_to_extract = pos_features + additional_features
print(features_to_extract)

['n_adj', 'n_adp', 'n_adv', 'n_aux', 'n_cconj', 'n_det', 'n_intj', 'n_noun', 'n_num', 'n_part', 'n_pron', 'n_propn', 'n_punct', 'n_sconj', 'n_sym', 'n_verb', 'n_space', 'n_uadj', 'n_uadp', 'n_uadv', 'n_uaux', 'n_ucconj', 'n_udet', 'n_uintj', 'n_unoun', 'n_unum', 'n_upart', 'n_upron', 'n_upropn', 'n_upunct', 'n_usconj', 'n_usym', 'n_uverb', 'n_uspace', 'a_word_ps', 'a_bry_ps', 'corr_ttr']


## Extract features 

To see how to extract features we will run a small text on a very small portion of the data (extracting the features for the full datasets would take a while (around 2-3 hours). 

*Task*:
(1) Retrieve the utterance texts for the first 100 utterances in each corpus and store them in a list.
(2) use spacy and a small model to process the utterances with the .pipe function (this is faster than processing each utterance individually). The result should be a list of processed spacy documents.
(3) initialize an extractor for each corpus with the list of processed spacy documents. use the extractor.extract(features=features_to_extract) function to extract the features for the processed utterances. The result is a list of dictionaries (one dictionary per utterance). Each dictionary contains the feature values for the respective utterance.
(4) print the first utterance to see how it looks like.




## Full Features

Since it takes too long for you to do the feature extraction for the assignment, I already ran the feature extraction for the full datasets and stored the results in two csv files in 'additional_data':

- features_genz.csv
- features_askoldpeople.csv

The files have the size of the utterance dataframes of each corpus after filtering - one row per utterance. Each column is a linguistic feature. Since we want to compare the two corpora based on these features, we normalize the raw counts:
- for n_pos and num_genz_words we divide by the number of tokens in the utterance (meta.num_tokens)
- for u_pos we divide by the number of corresponding n_pos feature (e.g. u_noun / n_noun) (to get the proportion of unique nouns out of all nouns used in the utterance)
- for the other features (a_word_ps, a_bry_ps, corr_ttr) no normalization is needed since they are already averages or ratios.

*Task*: Load the two csv files into pandas dataframes. Set the index to the same index as the utterance dataframes so that each utterance in the feature dataframes corresponds to the same utterance in the utterance dataframes. Add the features as meta information to the respective utterance objects in the two corpora. (You can iterate over the utterance objects, retrieve the utterance id, and then get the feature values from the dataframe based on the index). Normalize the feature values as described above.



The resulting meta data object should look like this:


Sample utterance meta information after adding features: ConvoKitMeta({'score': 1, 'top_level_comment': 'e2f2coo', 'retrieved_on': 1536079949, 'gilded': 0, 'gildings': None, 'subreddit': 'GenZ', 'stickied': False, 'permalink': '/r/GenZ/comments/8vr72c/what_kind_of_jobs_are_you_aiming_for/e2f2coo/', 'author_flair_text': '', 'parsed': [{'toks': [{'tok': 'Although'}, {'tok': 'it'}, {'tok': 'is'}, {'tok': 'very'}, {'tok': 'unlikely'}, {'tok': 'to'}, {'tok': 'happen'}, {'tok': 'because'}, {'tok': 'of'}, {'tok': 'where'}, {'tok': 'I'}, {'tok': 'live'}, {'tok': '('}, {'tok': 'buttfuck'}, {'tok': 'nowhere'}, {'tok': ','}, {'tok': 'Ohio'}, {'tok': ')'}, {'tok': 'I'}, {'tok': 'would'}, {'tok': 'like'}, {'tok': 'to'}, {'tok': 'go'}, {'tok': 'into'}, {'tok': '2d'}, {'tok': 'animation'}, {'tok': 'as'}, {'tok': 'a'}, {'tok': 'job'}, {'tok': '.'}]}], 'num_tokens': 30, 'num_genz_words': 0, 'n_adp': np.float64(0.1), 'n_adv': np.float64(0.06666666666666667), 'n_aux': np.float64(0.06666666666666667), 'n_cconj': np.float64(0.0), 'n_det': np.float64(0.03333333333333333), 'n_intj': np.float64(0.0), 'n_noun': np.float64(0.13333333333333333), 'n_num': np.float64(0.0), 'n_part': np.float64(0.06666666666666667), 'n_pron': np.float64(0.1), 'n_propn': np.float64(0.03333333333333333), 'n_punct': np.float64(0.13333333333333333), 'n_sconj': np.float64(0.1), 'n_sym': np.float64(0.0), 'n_verb': np.float64(0.13333333333333333), 'n_space': np.float64(0.0), 'n_uadj': np.float64(1.0), 'n_uadp': np.float64(0.3333333333333333), 'n_uadv': np.float64(0.5), 'n_uaux': np.float64(1.0), 'n_ucconj': 0.0, 'n_udet': np.float64(1.0), 'n_uintj': 0.0, 'n_unoun': np.float64(0.25), 'n_unum': 0.0, 'n_upart': np.float64(0.5), 'n_upron': np.float64(0.6666666666666666), 'n_upropn': np.float64(1.0), 'n_upunct': np.float64(0.25), 'n_usconj': np.float64(0.3333333333333333), 'n_usym': 0.0, 'n_uverb': np.float64(0.25), 'n_uspace': 0.0, 'a_word_ps': np.float64(15.0), 'a_bry_ps': np.float64(4.5), 'corr_ttr': np.float64(0.516), 'n_adj': np.float64(0.03333333333333333)})